# Machine Learning Zoomcamp

## 3. Machine Learning for Classification — Practice

Work through each exercise from memory before running it. If you get stuck, peek at the
course's own solution notebook: `notebook.ipynb` (`../notebook.ipynb`, one level up).

Dataset: [Telco customer churn](https://www.kaggle.com/blastchar/telco-customer-churn)

Plan:

* Data preparation
* Setting up the validation framework
* EDA
* Feature importance: churn rate and risk ratio
* Feature importance: mutual information
* Feature importance: correlation
* One-hot encoding
* Logistic regression
* Training logistic regression with Scikit-Learn
* Model interpretation
* Using the model


In [215]:
# Import pandas, numpy, matplotlib, seaborn under their usual aliases
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## 3.2 Data preparation

1. Download the Telco churn dataset from the URL below and read it into a DataFrame called `df`.
   `data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'`
2. Normalize the column names: lowercase them and replace spaces with underscores.
3. Find the columns with string (`object`) dtype and normalize their *values* too: lowercase and
   replace spaces with underscores.
4. `totalcharges` looks numeric but reads in as an object column. Coerce it to numeric with
   `pd.to_numeric(..., errors='coerce')` and check how many rows became `NaN`. Fill those with `0`.
5. Convert the `churn` target column from `yes`/`no` strings to `1`/`0` integers.

**Recall:** why does `errors='coerce'` matter here instead of just letting the conversion raise?


In [216]:
# 1. Download and read the dataset
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data)
df


In [217]:
# 2. Normalize column names
df.columns = df.columns.str.lower().str.replace(" ", "_")
df


In [218]:
# 3. Normalize string values in object columns
categorical = df.dtypes[df.dtypes == 'str'].index.to_list()
for c in categorical:
    df[c] = df[c].str.lower().str.replace(' ', '_')


In [219]:
# 4. Coerce totalcharges to numeric, check NaN count, fill with 0
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

df.isnull().sum()
df.totalcharges = df.totalcharges.fillna(0)
df.totalcharges.isnull().sum()


np.int64(0)


In [220]:
# 5. Convert churn to 1/0
df.churn = (df.churn == 'yes').astype(int)


## 3.3 Setting up the validation framework

1. Use `train_test_split` from `sklearn.model_selection` to split `df` into `df_full_train`
   (80%) and `df_test` (20%), with `random_state=1`.
2. Split `df_full_train` again into `df_train` (75%) and `df_val` (25%) — this yields an overall
   60/20/20 train/val/test split. Use `random_state=1` again.
3. Reset the index on all three DataFrames (`reset_index(drop=True)`).
4. Extract `y_train`, `y_val`, `y_test` as the `.values` of the `churn` column from each split.
5. Delete the `churn` column from `df_train`, `df_val`, `df_test` — why is this step necessary?

**Recall:** why do we split off the test set *before* splitting train/val, rather than doing a
single three-way split in one step?


In [221]:
# 1-2. Two-step split: full_train/test, then train/val
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 42)

len(df_train), len(df_val), len(df_test)


(4225, 1409, 1409)


In [222]:
# 3. Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)


In [223]:
# 4-5. Extract y arrays, then delete churn from the feature frames
y_train = df_train.churn
y_test = df_test.churn
y_val = df_val.churn
y_full_train = df_full_train.churn

del df_train['churn']
del df_test['churn']
del df_val['churn']
del df_full_train['churn']


## 3.4 EDA

1. Check `df_full_train.isnull().sum()` for missing values.
2. Look at the distribution of the target: `df_full_train.churn.value_counts(normalize=True)`.
3. Compute the global churn rate as `df_full_train.churn.mean()` and round it to 2 decimals.
4. Split the columns into `numerical` and `categorical` lists by hand (exclude `customerid` and
   `churn` from both).
5. For each categorical column, check `df_full_train[col].nunique()`.

**Recall:** why does the mean of a 0/1 column give you the churn rate directly?


In [224]:
# 1. Missing value check
df_full_train.isnull().sum()


customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
dtype: int64


In [225]:
# 2-3. Target distribution and global churn rate
df.churn.value_counts(normalize = True)

round(df.churn.mean(), 2)


np.float64(0.27)


In [226]:
# 4. Build numerical / categorical column lists
# print(df.dtypes)

numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity',
    'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv',
    'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod'
]


In [227]:
# 5. Unique value counts per categorical column
dic = {}
for c in categorical:
    dic[c] = (list(df[c].unique()))
length = 0
for key in dict:
    length += len(dict[key])
length
# df[(df['phoneservice'] == 'no') & ((df['multiplelines'] == 'no_phone_service') | (df['multiplelines'] == 'no'))]


43


In [228]:
df[numerical].shape


(7043, 3)


## 3.5 Feature importance: churn rate and risk ratio

1. Pick a categorical feature (e.g. `gender`). Group `df_full_train` by it and compute the mean
   churn rate per group with `.groupby('gender').churn.agg(['mean', 'count'])`.
2. For each group, compute the **difference** from the global churn rate (`group_mean - global`).
   A negative difference means that group churns *more* than average.
3. For the same feature, compute the **risk ratio**: `group_mean / global`. A ratio > 1 means the
   group is more likely to churn; < 1 means less likely.
4. Repeat steps 1-3 for `partner` and `contract` — which categories stand out as high risk?

**Recall:** in your own words, what's the difference between what the churn-rate-difference tells
you and what the risk ratio tells you, even though they're built from the same two numbers?


In [229]:
# 1. Group by gender: mean churn rate and count per group
churn_gender = df.groupby('gender').churn.agg(['mean', 'count'])


In [230]:
# 2. Difference from global churn rate
churn_global = df.churn.mean()

churn_female = df[df.gender == 'female'].churn.mean()
print("Female: " + str(float(churn_global) - float(churn_female)))

churn_male = df[df.gender == 'male'].churn.mean()
print("Male: " + str(float(churn_global) - float(churn_male)))


print(float(churn_global))


Female: -0.003838844802634356
Male: 0.0037664952662697093
0.2653698707936959


In [231]:
# 3. Risk ratio
risk_female = churn_female / churn_global
print(risk_female)

risk_male = churn_male / churn_global
print(risk_male)


1.014466016021912
0.9858066205669676


In [232]:
# 4. Repeat for partner and contract
df.groupby('partner').churn.agg(['mean', 'count'])

churn_partner = df[df['partner'] == 'yes'].churn.mean()
churn_no_partner = df[df['partner'] == 'no'].churn.mean()

print("Partner: " + str(churn_partner))
print("No Partner: " + str(churn_no_partner))

churn_partner/churn_global, churn_no_partner/churn_global


Partner: 0.1966490299823633
No Partner: 0.32957978577313923


(np.float64(0.7410375164075894), np.float64(1.241963847619165))


In [233]:
print(df.groupby('contract').churn.agg(['mean', 'count']))

churn_month_contract = df[df['contract'] == 'month-to-month'].churn.mean()
churn_year_contract = df[df['contract'] == 'one_year'].churn.mean()
churn_two_year_contract = df[df['contract'] == 'two_year'].churn.mean()

churn_month_contract/churn_global, churn_year_contract/churn_global, churn_two_year_contract/churn_global


                    mean  count
contract
month-to-month  0.427097   3875
one_year        0.112695   1473
two_year        0.028319   1695


(np.float64(1.609439583009717),
 np.float64(0.4246720984861445),
 np.float64(0.10671363703082902))


## 3.6 Feature importance: mutual information

1. Import `mutual_info_score` from `sklearn.metrics`.
2. Write a `mutual_info_churn_score(series)` function that computes
   `mutual_info_score(series, df_full_train.churn)`.
3. Apply it to every categorical column with `df_full_train[categorical].apply(...)`.
4. Sort the resulting scores descending (`.sort_values(ascending=False)`) — which categorical
   feature carries the most information about churn?

**Recall:** why can mutual information rank categorical features on one common scale, when churn
rate / risk ratio have to be inspected one feature (and one category) at a time?


In [234]:
# 1-2. Import mutual_info_score, define mutual_info_churn_score(series)
from sklearn.metrics import mutual_info_score


In [235]:
# 3-4. Apply across categorical columns, sort descending
mutual_info_dict = {}
for c in categorical:
    mutual_info_dict[c] = mutual_info_score(df.churn, df[c])
mutual_info_dict


{'gender': 3.7082914405128786e-05,
 'seniorcitizen': 0.010577263953987642,
 'partner': 0.011453657253317984,
 'dependents': 0.014467261139424592,
 'phoneservice': 7.215949186982484e-05,
 'multiplelines': 0.0008012658524292199,
 'internetservice': 0.05557418477268879,
 'onlinesecurity': 0.06467728245735829,
 'onlinebackup': 0.04679232253922637,
 'deviceprotection': 0.04391690927485155,
 'techsupport': 0.06302103606897548,
 'streamingtv': 0.031907975162527094,
 'streamingmovies': 0.03200094959522297,
 'contract': 0.09845305342598942,
 'paperlessbilling': 0.019194399646111526,
 'paymentmethod': 0.044518668630902994}


In [236]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df.churn)

mutual_info_churn = df[categorical].apply(mutual_info_churn_score)
mutual_info_churn.sort_values(ascending=False)


contract            0.098453
onlinesecurity      0.064677
techsupport         0.063021
internetservice     0.055574
onlinebackup        0.046792
paymentmethod       0.044519
deviceprotection    0.043917
streamingmovies     0.032001
streamingtv         0.031908
paperlessbilling    0.019194
dependents          0.014467
partner             0.011454
seniorcitizen       0.010577
multiplelines       0.000801
phoneservice        0.000072
gender              0.000037
dtype: float64


## 3.7 Feature importance: correlation

1. Compute `df_full_train[numerical].corrwith(df_full_train.churn)` — the correlation of each
   numerical feature with the churn target.
2. For each correlation value, classify it as LOW (`|r| < 0.2`), MEDIUM (`0.2 <= |r| < 0.5`), or
   STRONG (`|r| >= 0.5`).
3. Pick the numerical feature with the strongest correlation and split it into two groups at its
   median. Compare the churn rate of the two groups with `.groupby(...).churn.mean()` to sanity
   check the sign of the correlation.

**Recall:** why does correlation only apply to numerical features, and why did we need mutual
information for the categorical ones instead?


In [237]:
# 1. Correlation of numerical features with churn
corr_churn = df[numerical].corrwith(df.churn)
corr_churn


tenure           -0.352229
monthlycharges    0.193356
totalcharges     -0.198324
dtype: float64


In [238]:
# 2. Classify each correlation as LOW / MEDIUM / STRONG
for feat, r in corr_churn.items():
    abs_r = abs(r)
    label = "LOW" if abs(r) < 0.2 else "MEDIUM" if abs(r) < 0.5 else "STRONG"
    print("%s: %s" % (feat, label))


tenure: MEDIUM
monthlycharges: LOW
totalcharges: LOW


In [239]:
# 3. Median split + churn rate sanity check
df.tenure.median()

df_tenure_high = df[df['tenure'] > 29.0]
df_tenure_low = df[df['tenure'] <= 29.0]


## 3.8 One-hot encoding

1. Import `DictVectorizer` from `sklearn.feature_extraction`.
2. Convert `df_train[categorical + numerical]` to a list of dicts with
   `.to_dict(orient='records')`.
3. Fit a `DictVectorizer(sparse=False)` on the training dicts and transform them into `X_train`.
4. Inspect `dv.get_feature_names_out()` — how many columns did the categorical features expand
   into, and what happened to the numerical features?
5. Transform `df_val` the same way into `X_val`, re-using the **already-fitted** `dv` (don't
   re-fit on validation data).

**Recall:** why must `dv` be fit on the training data only, and reused (not refit) for validation
and test?


In [240]:
# 1-3. DictVectorizer: dicts -> fit_transform on df_train
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)


In [241]:
# 4. Inspect get_feature_names_out()
train_dict = df_train[categorical+numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical+numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)


In [242]:
# 5. Transform df_val with the already-fitted dv
X_train.shape


(4225, 45)


## 3.9-3.10 Logistic regression, trained with Scikit-Learn

1. Write a `sigmoid(z)` function: `1 / (1 + np.exp(-z))`. Plot it over `z = np.linspace(-7, 7, 51)`
   to see the S-curve.
2. Import `LogisticRegression` from `sklearn.linear_model`. Train it on `X_train`, `y_train` with
   `solver='liblinear'`, `random_state=1`.
3. Inspect `model.intercept_[0]` and `model.coef_[0]` — how many weights are there, and how does
   that compare to the number of columns in `X_train`?
4. Get hard predictions with `model.predict(X_val)` and soft predictions (probabilities) with
   `model.predict_proba(X_val)` — which column of the soft predictions corresponds to "churn"?
5. Threshold the soft predictions at 0.5 (`y_pred >= 0.5`) and compute accuracy as
   `(y_val == churn_decision).mean()`.

**Recall:** why does logistic regression need the sigmoid on top of the linear part, when plain
linear regression doesn't?


In [243]:
# 1. sigmoid(z) + plot
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

plt.plot(sigmoid(np.linspace(-7, 7, 51)))


In [244]:
# 2. Train LogisticRegression on X_train, y_train
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=10000)
repr(model.fit(X_train, y_train))


'LogisticRegression(max_iter=10000)'


In [245]:
X_train.shape


(4225, 45)


In [246]:
# 3. Inspect intercept_ and coef_
model.intercept_
model.coef_.round(3)


array([[ 0.752,  0.034, -0.787,  0.03 , -0.031,  0.075, -0.176,  0.1  ,
         0.093, -0.094, -0.45 ,  0.625, -0.176, -0.016, -0.171,  0.045,
         0.124,  0.125, -0.176,  0.05 ,  0.253, -0.176, -0.078, -0.18 ,
         0.179,  0.001, -0.002, -0.071, -0.136,  0.25 , -0.044,  0.045,
        -0.046,  0.149, -0.131, -0.176,  0.306, -0.059, -0.176,  0.234,
         0.206, -0.176, -0.031, -0.056,  0.   ]])


In [247]:
# 4. Hard vs soft predictions on X_val
y_pred = model.predict(X_val)
y_pred_proba = model.predict_proba(X_val)[:,1].round(3)
y_pred_proba


array([0.155, 0.265, 0.429, ..., 0.711, 0.039, 0.04 ], shape=(1409,))


In [248]:
# 5. Threshold at 0.5 and compute validation accuracy
accuracy = (y_pred == y_val).mean()

precision = ((y_pred == 1) & (y_val == 1)).sum() / (y_pred == 1).sum()

recall = ((y_pred == 1) & (y_val == 1)).sum() / ((((y_pred == 1) & (y_val == 1))) + ((y_pred == 0) & (y_val == 1))).sum()

accuracy, precision, recall


(np.float64(0.801277501774308),
 np.float64(0.6622950819672131),
 np.float64(0.5329815303430079))


In [249]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

print(precision_score(y_val, y_pred))
print(recall_score(y_val, y_pred))
print(accuracy_score(y_val, y_pred))


0.6622950819672131
0.5329815303430079
0.801277501774308


## 3.11 Model interpretation

1. `zip` the feature names (`dv.get_feature_names_out()`) with the model's weights
   (`model.coef_[0]`) into a dict, and print it sorted by absolute weight, descending.
2. Pick one categorical feature (e.g. `contract`) and look at only its one-hot weights — do they
   match the risk-ratio direction you found back in 3.5?
3. Train a smaller model using only `['contract', 'tenure', 'totalcharges']` and compare its
   validation accuracy to the full model — how much do we lose by dropping most features?

**Recall:** why does only one of the one-hot columns for a given categorical feature ever
contribute to a specific row's prediction?


In [266]:
# 1. zip feature names with weights, sort by |weight| descending
feature_names = dv.get_feature_names_out()

features_weight_dict = dict(zip(feature_names, model.coef_[0].round(5)))


In [271]:
# 2. Inspect one categorical feature's one-hot weights vs. its risk ratios
df.groupby('contract').churn.agg(['mean', 'count'])


In [275]:
churn_global = df.churn.mean()

features_weight_dict['contract=month-to-month'], features_weight_dict['contract=one_year'], features_weight_dict['contract=two_year']


(np.float64(0.75235), np.float64(0.03377), np.float64(-0.78713))


In [296]:
# 3. Small model on 3 features, compare validation accuracy
dv = DictVectorizer(sparse=False)
small_dict = df_train[['contract', 'tenure', 'totalcharges']].to_dict(orient='records')
X_small = dv.fit_transform(small_dict)

small_val_dict = df_val[['contract', 'tenure', 'totalcharges']].to_dict(orient='records')
X_val_small = dv.transform(small_val_dict)

repr(model.fit(X_small, y_train))

y_pred = model.predict(X_val_small)
(y_pred == y_val).mean()


np.float64(0.7636621717530163)


## 3.12 Using the model

1. Combine `df_train` and `df_val` into `df_full_train` (already have this from 3.3, but rebuild
   it explicitly here) and likewise concatenate `y_train`/`y_val` into `y_full_train`.
2. Fit `dv` and the logistic regression model on the combined full-train data.
3. Transform `df_test` with the refit `dv`, predict, and compute test accuracy.
4. Pick a single customer row from `df_test`, convert it to a dict, transform it with `dv`, and get
   the model's churn probability for that one customer. Compare to their actual label.

**Recall:** why is it standard practice to retrain on train+val for the final model, but still
evaluate on a test set that was never touched during any of this?


In [315]:
# 1. Rebuild df_full_train / y_full_train explicitly
df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)

y_full_train = pd.concat([y_train, y_val])


(5634,)


In [308]:
# 2. Fit dv + model on the combined full-train data
dv = DictVectorizer(sparse = False)
full_train_dict = df_full_train[categorical + numerical].to_dict(orient='records')

X_full_train = dv.fit_transform(full_train_dict)


In [320]:
# 3. Transform df_test, predict, compute test accuracy
test_dict = df_test[categorical + numerical].to_dict(orient='records')

X_test = dv.transform(test_dict)

model = LogisticRegression(max_iter=10000)

repr(model.fit(X_full_train, y_full_train))


In [341]:
# 4. Predict churn probability for a single test customer
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

test_accuracy = (y_pred == y_test).mean()
test_accuracy, recall_score(y_test, y_pred), precision_score(y_test, y_pred)

test_user = test_dict[16]
X_user = dv.transform(test_user)

user_prediction = model.predict(X_user)
user_prediction == y_test.iloc[16]


array([False])


## Wrap-up

In your own words (no code), answer:

1. Walk through why mutual information, risk ratio, and correlation each measure "feature
   importance" differently, and when you'd reach for which one.
2. What does `DictVectorizer` actually do to a row that has both numerical and categorical
   columns, and why do numerical columns pass through unchanged?
3. What's the practical difference between `model.predict(X)` and `model.predict_proba(X)`, and
   why might you want the soft predictions even if you're going to threshold them anyway?
